#### CREATE `CUSTOMERS_VW` FROM VOLUME
- CATALOG NAME: GIZMO
- SCHEMA NAME: BRONZE
- VIEW NAME: CUSTOMERS_VW

In [0]:
%python
customers_df = (spark.read.format('json').load('/Volumes/gizmo/landing/operational_data/customers/'))
# display(customers_df.limit(10))
print(f'Row Count: {customers_df.count()}')

In [0]:
CREATE OR REPLACE VIEW GIZMO.BRONZE.CUSTOMERS_VW
AS
SELECT
customer_id,
customer_name,
to_date(date_of_birth,'yyyy-MM-dd') as date_of_birth,
email,
to_date(member_since,'yyyy-MM-dd') as member_since,
telephone,
to_timestamp(created_timestamp,'yyyy-MM-dd HH:mm:ss') as created_timestamp,
_metadata.file_path as file_path,
_metadata.file_name as file_name,
current_timestamp() as load_timestamp
 FROM json.`/Volumes/gizmo/landing/operational_data/customers/`;

#### QUERY `CUSTOMERS_VW` TO VALIDATE THE DATA

In [0]:
SELECT * FROM GIZMO.BRONZE.CUSTOMERS_VW;

In [0]:
%python
customers_count_df = spark.sql('''SELECT * FROM GIZMO.BRONZE.CUSTOMERS_VW''');
print(f'Row Count: {customers_count_df.count()}')

#### BELOW COMMAND TO EXECUTE THE FUNCTION

In [0]:
%run /Workspace/Users/pde1409@hotmail.com/AzureDatabricks-Gizmo/AzureDatabricks-GizmoBox/01.GizmoBox/02.Config/01.config.py

In [0]:
%python
try:
    verify_pipeline_counts(customers_df, customers_count_df, "01.IngestCustomersJSON")
except AssertionError as e:
    # This ensures the notebook actually fails if scheduled via a Databricks Workflow/Job
    raise e

In [0]:
%skip
%python
spark.sql("""
CREATE OR REPLACE TABLE GIZMO.BRONZE.INGEST_LOGS (
  log_id STRING,
  event_time TIMESTAMP,
  event_type STRING,
  source_table STRING,
  target_table STRING,
  record_count BIGINT,
  status STRING,
  message STRING,
  user_name STRING,
  notebook_path STRING,
  pipeline_name STRING
)
COMMENT 'Logging table for data ingestion events and pipeline activity'
""")

#### CAPTURE AUDIT / OBSERVABILITY MECHANISM 

In [0]:
%python
from pyspark.sql import Row
from datetime import datetime
import uuid

load_start_time = datetime.now()
record_count = customers_count_df.count()
load_end_time = datetime.now()

user_name = 'pde1409'
log_entry = Row(
    log_id=str(uuid.uuid4()),
    event_time=datetime.now(),
    event_type="LOAD",
    source_table="json.`/Volumes/gizmo/landing/operational_data/customers/`",
    target_table="GIZMO.BRONZE.CUSTOMERS_VW",
    record_count=record_count,
    status="SUCCESS",
    message="Loaded customers data into bronze view",
    user_name=user_name,
    notebook_path=dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get(),
    pipeline_name='01.IngestCustomersJSON',
    load_start_time=load_start_time,
    load_end_time=load_end_time
)

In [0]:
%python
log_entry_df = spark.createDataFrame([log_entry])
log_entry_df.write.mode("append").saveAsTable("GIZMO.BRONZE.AUDIT_LOGS")

In [0]:
%python
dbutils.notebook.exit("CUSTOMERS HAS BEEN LOADED SUCCESSFULLY INTO GIZMO.BRONZE.CUSTOMERS_VW")

In [0]:
SELECT * FROM GIZMO.BRONZE.AUDIT_LOGS;